# AUTOMATITZAR DICCIONARI FONÈTIC 

In [18]:
import os
import sys
import json
import time
from openai import OpenAI
from pathlib import Path
import configparser

# CONFIGURACIÓ
MODEL = 'gpt-4o'
IDIOMA_TTS = 'Castellano' #'Catalán' 'Euskera'
MAX_OUTPUT_TOKENS = 2048

# Preus per a càlcul de cost (USD per MToken) 
PRICE = {
    'gpt-4o-mini':      {'in': 0.150, 'cached_in': 0.075, 'out': 0.600},
    'gpt-4o':           {'in': 2.500, 'cached_in': 1.250, 'out': 10.000},
}

# Inicialització del client
key = os.environ.get("OPENAI_API_KEY")
font = "env:OPENAI_API_KEY"
cfg_path = "/media/ugiat/dd2/projects/nerea/sintetic_dataset/Sintetic-dataset/utils/config.ini"
cfg = configparser.ConfigParser()
cfg.read(cfg_path)
if 'OPENAI' in cfg and 'KEY' in cfg['OPENAI']:
    key = cfg['OPENAI']['KEY']
    font = f"config.ini:{cfg_path}"
if not key:
    raise RuntimeError("No s'ha trobat cap OPENAI_API_KEY ni a env ni a config.ini")

base_url = os.environ.get("OPENAI_BASE_URL")  # opcional: servidors locals
if base_url:
    font += f" (base_url={base_url})"
client = OpenAI(api_key=key, base_url=base_url)

In [ ]:
# Nom intern de l'esquema (visible al model i als logs) 
SCHEMA_NAME = "phonetic_dictionary_generation"

# Esquema estricte adaptat per a llistes d'entitats (Strict = True) 
JSON_SCHEMA = {
    "type": "object",
    "properties": {
        "diccionari": {
            "type": "array",
            "items": {
                "type": "object",
                "properties": {
                    "entitat_original": {
                        "type": "string",
                        "description": "El nom de l'entitat tal com apareix en el text original."
                    },
                    "transcripcio_fonetica": {
                        "type": "string",
                        "description": "La reescriptura fonètica segons les regles establertes."
                    }
                },
                "required": ["entitat_original", "transcripcio_fonetica"],
                "additionalProperties": False
            }
        }
    },
    "required": ["diccionari"],
    "additionalProperties": False
}

# Particularidades ortogràfiques/d'accentuació per idioma objectiu.
# Nota: només tenim exemples verificats per Castellano (el que estem executant ara).
# Per Catalán/Euskera, el prompt indica el PRINCIPI a aplicar però no inventa exemples
# concrets sense validació manual (accentuació i fonotàctica difereixen molt entre idiomes).
LANGUAGE_NOTES = {
    'Castellano': (
        "Usa tildes según las reglas de acentuación del castellano cuando la pronunciación real "
        "no coincida con la acentuación por defecto (p.ej. una palabra extranjera que termina en "
        "consonante distinta de 'n'/'s' pero se pronuncia aguda)."
    ),
    'Catalán': (
        "Usa las convenciones ortográficas y de acentuación propias del catalán (accents oberts/"
        "tancats, dígrafos catalanes). No apliques reglas de acentuación castellanas."
    ),
    'Euskera': (
        "El euskera batua no usa tildes en su ortografía nativa: NO añadas acentos gráficos. "
        "Usa los dígrafos propios del euskera (tx, tz, ts, x, j, k...) en vez de convenciones "
        "ortográficas castellanas para representar sonidos extranjeros."
    ),
}

EXAMPLES = {
    'Castellano': [
        ("Wall Street", "Guol estrit"),
        ("Lamine Yamal", "Lamín Yamal"),   # sílaba final tónica: sin tilde un TTS castellano la leería llana
        ("Mbappé", "Embapé"),               # "Mb" inicial no es pronunciable en castellano -> vocal de apoyo
        ("Junior", "Yúnior"),  
    ],
}

def build_examples_block(idioma_tts: str) -> str:
    ejemplos = EXAMPLES.get(idioma_tts)
    if not ejemplos:
        return (
            f"No se incluyen ejemplos verificados para {idioma_tts} todavía: aplica los mismos "
            f"principios (respelling literal, vocal de apoyo en grupos consonánticos imposibles, "
            f"expansión de símbolos) usando exclusivamente la ortografía y fonética propias de "
            f"{idioma_tts}, sin recurrir a convenciones castellanas."
        )
    lineas = "\n".join(f'- "{orig}" -> "{fon}"' for orig, fon in ejemplos)
    return f"Ejemplos orientativos (ajusta si difieren de lo que oigas realmente en el TTS):\n{lineas}"

def build_system_prompt(idioma_tts: str) -> str:
    nota_idioma = LANGUAGE_NOTES.get(
        idioma_tts,
        f"Usa las convenciones ortográficas y de acentuación propias de {idioma_tts}."
    )
    ejemplos_block = build_examples_block(idioma_tts)
    return f"""Eres un lingüista experto en fonética y sistemas Text-to-Speech (TTS).
Vas a recibir una lista de entidades (nombres propios, acrónimos, extranjerismos, símbolos).
Tu objetivo es devolver su adaptación fonética para que un modelo TTS configurado en {idioma_tts}
lo lea correctamente con naturalidad, como un presentador de noticias.

Reglas de integridad (obligatorias):
- Debes devolver exactamente una entrada por cada entidad de entrada, ni más ni menos.
- Conserva el orden de la lista de entrada.
- Nunca traduzcas, abrevies, inventes, omitas ni dividas una entidad en varias.
- Si una entidad ya se pronunciaría correctamente tal cual, devuélvela sin modificar (como mucho corrige mayúsculas o tildes que falten).

Reglas de reescritura fonética:
1. Siglas deletreadas (ej. UE, FMI): separa cada letra con un espacio y mantenlas en mayúscula. (Resultado: "U E", "F M I").
2. Acrónimos léxicos (ej. PSOE, OTAN): capitaliza solo la primera letra y ajusta la acentuación si la pronunciación real no coincide con la acentuación por defecto de {idioma_tts}. (Resultado: "Psoe", "Otán").
3. Extranjerismos y nombres propios complejos: aplica respelling fonético usando la ortografía literal de {idioma_tts}, de forma que alguien sin conocimiento del idioma de origen se acerque a la pronunciación real. Si la palabra empieza por un grupo consonántico imposible en {idioma_tts} (ej. "Mb", "Ng", "Pf"), añade una vocal de apoyo o simplifica el grupo para que sea pronunciable.
4. Símbolos y unidades (ej. ºC, %, km/h): expándelos a su forma leída completa en palabras de {idioma_tts}. (Resultado en castellano: "grados Celsius", "por ciento", "kilómetros por hora").

Particularidades de {idioma_tts}: {nota_idioma}

{ejemplos_block}
"""

SYSTEM_PROMPT = build_system_prompt(IDIOMA_TTS)


In [20]:
# %%
import random
from openai import RateLimitError, APIConnectionError, APITimeoutError, InternalServerError

def cridar_openai_sdk(entitats, model=MODEL):
    """
    Crida a /v1/chat/completions amb Structured Outputs forçats .
    """
    prompt_usuari = f"Entitats a processar:\n{json.dumps(entitats)}"
    t0 = time.time()
    
    resposta = client.chat.completions.create(
        model=model,
        max_tokens=MAX_OUTPUT_TOKENS,
        temperature=0, # Determinisme per a resultats estables 
        seed=42,       # Seed fixat per a reproductibilitat 
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": prompt_usuari},
        ],
        response_format={
            "type": "json_schema",
            "json_schema": {
                "name": SCHEMA_NAME,
                "strict": True,
                "schema": JSON_SCHEMA,
            },
        },
    )

    msg = resposta.choices[0].message
    
    # Control de rebuig del model per seguretat o esquema 
    if getattr(msg, "refusal", None):
        raise RuntimeError(f"Model refusal: {msg.refusal}")

    resultat_obj = json.loads(msg.content)
    usage = resposta.usage
    cached = getattr(usage.prompt_tokens_details, "cached_tokens", 0) if getattr(usage, "prompt_tokens_details", None) else 0

    meta = {
        "duration_s": round(time.time() - t0, 2),
        "prompt_tokens": usage.prompt_tokens,
        "completion_tokens": usage.completion_tokens,
        "cached_tokens": cached,
        "finish_reason": resposta.choices[0].finish_reason,
    }
    return resultat_obj, meta

def cost_chunk(meta, model=MODEL):
    """Calcula el cost en USD d'una crida, tenint en compte els tokens cached ."""
    p = PRICE.get(model)
    if not p:
        return None
    non_cached = max(0, meta['prompt_tokens'] - meta['cached_tokens'])
    return (
        non_cached * p['in'] / 1_000_000
        + meta['cached_tokens'] * p['cached_in'] / 1_000_000
        + meta['completion_tokens'] * p['out'] / 1_000_000
    )

def validar_diccionari(resultat, entitats_input):
    """
    Valida la integritat del diccionari retornat pel model respecte a l'input:
    mateix nombre d'entrades, mateixes entitats (sense inventar-ne ni ometre'n) i
    cap transcripció buida. Llança ValueError amb el detall si alguna cosa falla.
    """
    diccionari = resultat.get("diccionari", [])
    originals_output = [item["entitat_original"] for item in diccionari]

    if len(diccionari) != len(entitats_input):
        raise ValueError(
            f"El model ha retornat {len(diccionari)} entrades per a {len(entitats_input)} "
            f"entitats d'entrada."
        )

    if set(originals_output) != set(entitats_input):
        faltants = set(entitats_input) - set(originals_output)
        sobrants = set(originals_output) - set(entitats_input)
        raise ValueError(
            "Les entitats retornades no coincideixen amb les d'entrada. "
            f"Faltants: {sorted(faltants) or '—'} | Inventades: {sorted(sobrants) or '—'}"
        )

    if originals_output != entitats_input:
        raise ValueError(
            "Les entitats retornades no respecten l'ordre de l'input "
            "(mateix conjunt, ordre diferent)."
        )

    buides = [item["entitat_original"] for item in diccionari if not item["transcripcio_fonetica"].strip()]
    if buides:
        raise ValueError(f"Transcripció fonètica buida per a: {buides}")

    return True

ERRORS_TRANSITORIS = (RateLimitError, APIConnectionError, APITimeoutError, InternalServerError)

def cridar_amb_reintents(entitats, model=MODEL, max_reintents=4):
    """Reintenta amb backoff exponencial + jitter davant errors transitoris de l'API."""
    for intent in range(1, max_reintents + 1):
        try:
            return cridar_openai_sdk(entitats, model=model)
        except ERRORS_TRANSITORIS as e:
            if intent == max_reintents:
                raise
            espera = (2 ** (intent - 1)) + random.uniform(0, 1)
            print(f"  Avís: {type(e).__name__} (intent {intent}/{max_reintents}). Reintentant en {espera:.1f}s...")
            time.sleep(espera)

def deduplicar_preservant_ordre(elements):
    vist = set()
    resultat = []
    for e in elements:
        if e not in vist:
            vist.add(e)
            resultat.append(e)
    return resultat

def processar_entitats(entitats_input, mida_lot=40, model=MODEL):
    """
    Processa una llista d'entitats en lots (per no sobrepassar MAX_OUTPUT_TOKENS ni
    fer crides massa grans), amb deduplicació prèvia, reintents davant errors
    transitoris i validació d'integritat per lot. Retorna el diccionari fonètic
    complet i les metadades agregades (tokens, cost, durada).
    """
    entitats_uniques = deduplicar_preservant_ordre(entitats_input)
    lots = [entitats_uniques[i:i + mida_lot] for i in range(0, len(entitats_uniques), mida_lot)]

    diccionari_fonetic = {}
    metadades_totals = {"duration_s": 0.0, "prompt_tokens": 0, "completion_tokens": 0, "cached_tokens": 0}
    cost_total = 0.0

    for idx, lot in enumerate(lots, start=1):
        print(f"Lot {idx}/{len(lots)} ({len(lot)} entitats)...")
        resultat, meta = cridar_amb_reintents(lot, model=model)
        validar_diccionari(resultat, lot)

        for item in resultat["diccionari"]:
            diccionari_fonetic[item["entitat_original"]] = item["transcripcio_fonetica"]

        for k in ("duration_s", "prompt_tokens", "completion_tokens", "cached_tokens"):
            metadades_totals[k] += meta[k]
        cost_lot = cost_chunk(meta, model=model)
        if cost_lot is not None:
            cost_total += cost_lot

    metadades_totals["cost_usd"] = round(cost_total, 6)
    return diccionari_fonetic, metadades_totals

In [21]:
# %%
# Entitats a processar: Font A (ja decidida, entra sempre al dataset) + Font B
# (candidates pendents de veredicte -- el round-trip de `src/verify_entities.py`
# les usarà per decidir si hi entren de veritat, i necessita aquesta fonètica per fer-ho).
#
# Generar-les juntes en un sol pas evita duplicar la crida LLM: la Font B nomes es
# torna a demanar si canvia la llista de candidates, no cada cop que es verifica.
ROOT = Path("/media/ugiat/dd2/projects/nerea/sintetic_dataset/Sintetic-dataset")

with open(ROOT / "lab/entitats/entidades_candidatas.json", encoding="utf-8") as f:
    entitats_font_a = list(json.load(f).keys())

with open(ROOT / "lab/entitats/entidades_fuente_b_validadas.json", encoding="utf-8") as f:
    _font_b = json.load(f)
entitats_font_b = [v["grafia_correcta"] for v in _font_b.get("validadas", [])]
entitats_font_b += [v["grafia_correcta"] for v in _font_b.get("rechazadas", []) if v["tipo"] != "NO_ENTIDAD"]

entitats_input = list(dict.fromkeys(entitats_font_a + entitats_font_b))  # deduplicat preservant ordre
print(f"Font A: {len(entitats_font_a)} | Font B: {len(entitats_font_b)} | total únic: {len(entitats_input)}")

print(f"Processant {len(entitats_input)} entitats...")

try:
    diccionari_fonetic, metadades = processar_entitats(entitats_input, mida_lot=40)

    print("\n--- RESUM DE L'EXECUCIÓ ---")
    print(f"Durada total: {metadades['duration_s']:.2f}s")
    print(f"Tokens: Prompt={metadades['prompt_tokens']} | Completion={metadades['completion_tokens']}")
    print(f"Cost estimat: ${metadades['cost_usd']:.6f}")

    print("\n--- DICCIONARI GENERAT ---")
    print(json.dumps(diccionari_fonetic, indent=2, ensure_ascii=False))

except Exception as e:
    print(f"Error durant la inferència: {e}")

Processant 24 entitats...
Lot 1/1 (24 entitats)...

--- RESUM DE L'EXECUCIÓ ---
Durada total: 4.54s
Tokens: Prompt=760 | Completion=432
Cost estimat: $0.006220

--- DICCIONARI GENERAT ---
{
  "Lamine Yamal": "Lamín Yamal",
  "UE": "U E",
  "Transfermarkt": "Transfermarkt",
  "OSCAN": "Oscán",
  "OpenAI": "Open AI",
  "Hollywood": "Jólivuud",
  "Prime Time": "Praim Taim",
  "Startup": "Startap",
  "Hacker": "Jáker",
  "New York": "Niu York",
  "PSOE": "Psoe",
  "Washington": "Guáshington",
  "ECA": "E C A",
  "Junior": "Júnior",
  "FMI": "F M I",
  "DatosRTVE": "Datos R T V E",
  "aemet": "Aemet",
  "ºC": "grados Celsius",
  "feijoo": "Feijóo",
  "Mbappé": "Embapé",
  "OTAN": "Otán",
  "NASA": "Nasa",
  "km/h": "kilómetros por hora",
  "%": "por ciento"
}


In [22]:
# %%
# Guardar l'output per poder fer la revisió manual abans de passar-ho al Pas B.
#
# Ruta canonica: es la mateixa que fa servir `generate_sentences.ipynb` Fase 3
# (DICCIONARI_PRINCIPAL) i que llegira `src/verify_entities.py`. Abans aquest fitxer
# es guardava al directori d'execucio (sense ruta fixa), cosa que el desconnectava
# de la resta del pipeline.
output_path = ROOT / f"lab/entitats/diccionaris/diccionari_fonetic_{IDIOMA_TTS.lower()}.json"
output_path.parent.mkdir(parents=True, exist_ok=True)

with open(output_path, 'w', encoding='utf-8') as f:
    json.dump(diccionari_fonetic, f, indent=2, ensure_ascii=False)

print(f"Diccionari exportat a {output_path}. REVISIÓ MANUAL REQUERIDA abans d'integrar-ho a la pipeline.")

Diccionari exportat a diccionari_fonetic_castellano.json. REVISIÓ MANUAL REQUERIDA abans d'integrar-ho a la pipeline.
